In [ ]:
# Cell 1 — Environment, paths, and dependencies

import importlib.util
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path

# ------------------------------------------------------------
# Detect Google Colab
# ------------------------------------------------------------

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f"Environment: {'Google Colab' if IS_COLAB else 'Local Jupyter / Python'}")

# ------------------------------------------------------------
# Pin the ATLAS content version this notebook fetches.
#
# None means "use the latest release listed in dist/manifest.yaml".
# Set to a release string (e.g. "2026.08") to pin: the notebook then
# resolves that release to its versioned dist file via manifest.yaml
# (v6/ATLAS-<version>.yaml, or legacy/ATLAS-<version>.yaml for older
# format versions) and raises if that release is not listed.
# ------------------------------------------------------------

PIN = None

# ------------------------------------------------------------
# Define the security_frameworks directory
# ------------------------------------------------------------

if IS_COLAB:
    # Colab's working area
    SECURITY_FRAMEWORKS_DIR = Path("/content/security_frameworks")

else:
    # When running locally, the notebook must be launched from the
    # security_frameworks directory — guard against an accidental
    # launch from elsewhere, which would otherwise silently create
    # MITRE_ATLAS/ and MITRE_ATLAS_output/ in the wrong place.
    SECURITY_FRAMEWORKS_DIR = Path.cwd()

    if SECURITY_FRAMEWORKS_DIR.name != "security_frameworks":
        raise RuntimeError(
            "This notebook must be launched from the security_frameworks "
            f"directory. Current working directory: {SECURITY_FRAMEWORKS_DIR}"
        )

# Create the directory if necessary
SECURITY_FRAMEWORKS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Repository and output locations
# ------------------------------------------------------------

REPO_DIR = SECURITY_FRAMEWORKS_DIR / "MITRE_ATLAS"
OUTPUT_DIR = SECURITY_FRAMEWORKS_DIR / "MITRE_ATLAS_output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/mitre-atlas/atlas-data.git"

print(f"Security frameworks directory: {SECURITY_FRAMEWORKS_DIR}")
print(f"Repository:                    {REPO_DIR}")
print(f"Output directory:              {OUTPUT_DIR}")
print(f"Pinned version:                {PIN if PIN else '(none — latest)'}")

# ------------------------------------------------------------
# Check Git
# ------------------------------------------------------------

GIT = shutil.which("git")

if GIT is None:
    raise RuntimeError(
        "Git is not installed or is not available on PATH."
    )

print(f"\n✓ Git: {GIT}")

# ------------------------------------------------------------
# Check/install uv
# ------------------------------------------------------------

UV = shutil.which("uv")

if UV is None:
    # Look for uv next to the running interpreter first (covers a venv
    # where uv was installed into Scripts/bin but PATH was not updated).
    for scripts_dir in {
        Path(sysconfig.get_path("scripts")),
        Path(sys.executable).parent,
    }:
        candidate = scripts_dir / ("uv.exe" if sys.platform == "win32" else "uv")

        if candidate.exists():
            UV = str(candidate)
            break

if UV is None:
    print("\nuv is not installed. Installing uv...")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "uv"],
        check=True,
    )

    UV = shutil.which("uv")

    if UV is None:
        for scripts_dir in {
            Path(sysconfig.get_path("scripts")),
            Path(sys.executable).parent,
        }:
            candidate = scripts_dir / (
                "uv.exe" if sys.platform == "win32" else "uv"
            )

            if candidate.exists():
                UV = str(candidate)
                break

if UV is None:
    raise RuntimeError(
        "uv installation completed, but the uv executable could not be found."
    )

print(f"✓ uv: {UV}")

# ------------------------------------------------------------
# Check/install PyYAML for the notebook kernel
# ------------------------------------------------------------

if importlib.util.find_spec("yaml") is None:
    print("\nPyYAML is not installed. Installing PyYAML...")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "PyYAML"],
        check=True,
    )

    if importlib.util.find_spec("yaml") is None:
        raise RuntimeError(
            "PyYAML installation completed, but the yaml module is still "
            "not importable."
        )

print("✓ PyYAML is available")

import yaml  # noqa: E402  (needed by later cells; confirmed importable above)

In [ ]:
# Cell 2 — Clone or update MITRE ATLAS Data

if not (REPO_DIR / ".git").exists():

    print("Downloading MITRE ATLAS Data...")

    subprocess.run(
        [
            GIT,
            "clone",
            "--depth",
            "1",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )

else:

    print("MITRE ATLAS repository already exists. Updating...")

    subprocess.run(
        [
            GIT,
            "-C",
            str(REPO_DIR),
            "fetch",
            "origin",
            "--depth",
            "1",
        ],
        check=True,
    )

    # Determine the repository's default branch.
    result = subprocess.run(
        [
            GIT,
            "-C",
            str(REPO_DIR),
            "symbolic-ref",
            "--quiet",
            "--short",
            "refs/remotes/origin/HEAD",
        ],
        capture_output=True,
        text=True,
        check=False,
    )

    remote_head = result.stdout.strip()

    if not remote_head:
        remote_head = "origin/main"

    subprocess.run(
        [
            GIT,
            "-C",
            str(REPO_DIR),
            "reset",
            "--hard",
            remote_head,
        ],
        check=True,
    )

# ------------------------------------------------------------
# Synchronize dependencies
# ------------------------------------------------------------

print("\nSynchronizing MITRE ATLAS dependencies...")

subprocess.run(
    [UV, "sync"],
    cwd=str(REPO_DIR),
    check=True,
)

# ------------------------------------------------------------
# Record exact Git commit
# ------------------------------------------------------------

commit_result = subprocess.run(
    [
        GIT,
        "-C",
        str(REPO_DIR),
        "rev-parse",
        "HEAD",
    ],
    capture_output=True,
    text=True,
    check=True,
)

git_commit = commit_result.stdout.strip()

print("\n" + "=" * 70)
print("MITRE ATLAS repository ready")
print("=" * 70)
print(f"Repository : {REPO_DIR}")
print(f"Git commit : {git_commit}")
print(f"Output     : {OUTPUT_DIR}")

In [ ]:
# Cell 3 — Generate versioned ATLAS STIX bundles + manifest

import hashlib
import json
from datetime import datetime, timezone

# ------------------------------------------------------------
# Resolve the ATLAS release to fetch via dist/manifest.yaml.
#
# manifest.yaml is the authoritative release index: each entry maps a
# release string to the dist file(s) that hold it, newest first. This
# is used instead of the "ATLAS-latest.yaml" symlinks because a Windows
# checkout without symlink support leaves those as one-line text
# placeholders rather than real YAML.
# ------------------------------------------------------------

manifest_path = REPO_DIR / "dist" / "manifest.yaml"

if not manifest_path.exists():
    raise FileNotFoundError(f"ATLAS release manifest not found: {manifest_path}")

with manifest_path.open("r", encoding="utf-8") as f:
    release_index = yaml.safe_load(f)

if not release_index:
    raise ValueError(f"ATLAS release manifest is empty: {manifest_path}")

if PIN is None:
    release_entry = release_index[0]
else:
    release_entry = next(
        (entry for entry in release_index if str(entry.get("release")) == str(PIN)),
        None,
    )

    if release_entry is None:
        available = ", ".join(str(entry.get("release")) for entry in release_index)
        raise ValueError(
            f"Pinned ATLAS release {PIN!r} was not found in the release "
            f"manifest. Available releases: {available}"
        )

release_versions = release_entry.get("versions") or []

if not release_versions:
    raise ValueError(
        f"ATLAS release {release_entry.get('release')!r} lists no dist files."
    )

# Prefer the highest-numbered format-version dist file for the release
# (matches current "latest" behaviour, which reads the v6 file).
release_version_entry = max(
    release_versions, key=lambda v: str(v.get("format-version", ""))
)

input_file = REPO_DIR / "dist" / release_version_entry["path"]

if not input_file.exists():
    raise FileNotFoundError(
        f"ATLAS dist file listed in manifest.yaml does not exist: {input_file}"
    )

print("ATLAS input:")
print(f"  {input_file}")

# ------------------------------------------------------------
# Read metadata
# ------------------------------------------------------------

with input_file.open("r", encoding="utf-8") as f:
    atlas_data = yaml.safe_load(f)

if not isinstance(atlas_data, dict):
    raise ValueError(f"ATLAS YAML did not parse to a mapping: {input_file}")

collection = atlas_data.get("collection")

if not isinstance(collection, dict) or "version" not in collection:
    raise KeyError(f"'collection.version' is missing from {input_file}")

content_version = collection["version"]

if "format-version" not in atlas_data:
    raise KeyError(f"'format-version' is missing from {input_file}")

format_version = atlas_data["format-version"]

print(f"\nATLAS content version : {content_version}")
print(f"ATLAS format version  : {format_version}")

# ------------------------------------------------------------
# Output locations: versioned + flat "-latest" copies
# ------------------------------------------------------------

VERSION_DIR = OUTPUT_DIR / str(content_version)
VERSION_DIR.mkdir(parents=True, exist_ok=True)

atlas_stix_versioned = VERSION_DIR / f"ATLAS-STIX-{content_version}.json"
atlas_attack_stix_versioned = (
    VERSION_DIR / f"ATLAS-ATTACK-Enterprise-STIX-{content_version}.json"
)
source_yaml_versioned = VERSION_DIR / input_file.name

atlas_stix_latest = OUTPUT_DIR / "ATLAS-STIX-latest.json"
atlas_attack_stix_latest = OUTPUT_DIR / "ATLAS-ATTACK-Enterprise-STIX-latest.json"

generator = REPO_DIR / "tools" / "atlas_to_stix.py"

if not generator.exists():
    raise FileNotFoundError(f"STIX generator not found: {generator}")

# Keep a copy of the exact source YAML alongside its generated bundles.
shutil.copy2(input_file, source_yaml_versioned)

# ------------------------------------------------------------
# Generate ATLAS STIX
# ------------------------------------------------------------

print("\nGenerating ATLAS STIX...")

subprocess.run(
    [UV, "run", str(generator), "-i", str(input_file), "-o", str(atlas_stix_versioned)],
    cwd=str(REPO_DIR),
    check=True,
)

print("✓ ATLAS STIX generated:")
print(f"  {atlas_stix_versioned}")

# ------------------------------------------------------------
# Generate ATLAS + ATT&CK Enterprise STIX
# ------------------------------------------------------------

print("\nGenerating ATLAS + ATT&CK Enterprise STIX bundle...")

subprocess.run(
    [
        UV,
        "run",
        str(generator),
        "-i",
        str(input_file),
        "--include-attack",
        "-o",
        str(atlas_attack_stix_versioned),
    ],
    cwd=str(REPO_DIR),
    check=True,
)

print("✓ ATLAS + ATT&CK Enterprise STIX generated:")
print(f"  {atlas_attack_stix_versioned}")

# ------------------------------------------------------------
# Verify the generator actually produced non-empty content, then
# publish the flat "-latest" copies.
# ------------------------------------------------------------

with atlas_stix_versioned.open("r", encoding="utf-8") as f:
    atlas_stix_data = json.load(f)

technique_count = sum(
    1
    for obj in atlas_stix_data.get("objects", [])
    if obj.get("type") == "attack-pattern"
)

if technique_count == 0:
    raise ValueError(
        f"Generated ATLAS STIX bundle contains zero techniques: {atlas_stix_versioned}"
    )

shutil.copy2(atlas_stix_versioned, atlas_stix_latest)
shutil.copy2(atlas_attack_stix_versioned, atlas_attack_stix_latest)

# ------------------------------------------------------------
# Manifest: repo, commit, pin, versions, retrieval time, file
# hashes/counts, and the full id lists per object type.
#
# ATLAS has no deprecation marker, so a vanished id between releases
# can only be found by diffing these id lists across manifest.json
# versions — hence the id lists are recorded in full, not just counts.
# ------------------------------------------------------------


def sha256_of(path):
    digest = hashlib.sha256()

    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)

    return digest.hexdigest()


id_lists = {
    "tactics": sorted((atlas_data.get("tactics") or {}).keys()),
    "techniques": sorted((atlas_data.get("techniques") or {}).keys()),
    "mitigations": sorted((atlas_data.get("mitigations") or {}).keys()),
    "case_studies": sorted((atlas_data.get("case-studies") or {}).keys()),
}

counts = {key: len(ids) for key, ids in id_lists.items()}

if counts["techniques"] == 0:
    raise ValueError(f"ATLAS YAML lists zero techniques: {input_file}")

manifest_files = {
    "source_yaml": source_yaml_versioned,
    "atlas_stix": atlas_stix_versioned,
    "atlas_attack_stix": atlas_attack_stix_versioned,
}

manifest = {
    "repository": REPO_URL,
    "git_commit": git_commit,
    "pin": PIN,
    "content_version": content_version,
    "format_version": format_version,
    "retrieved_at": datetime.now(timezone.utc).isoformat(),
    "files": {
        name: {
            "path": str(path.relative_to(OUTPUT_DIR)),
            "sha256": sha256_of(path),
        }
        for name, path in manifest_files.items()
    },
    "counts": counts,
    "ids": id_lists,
}

manifest_path_out = OUTPUT_DIR / "manifest.json"

with manifest_path_out.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False, sort_keys=True)
    f.write("\n")

print(f"\n✓ Manifest written: {manifest_path_out}")

# ------------------------------------------------------------
# Summary — deliberately prints everything needed to verify a run
# (version, snapshot folder, counts, per-file sha256) without needing
# to open any file under MITRE_ATLAS_output/. The full id lists stay
# in manifest.json only; they are too long to print here.
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MITRE ATLAS STIX GENERATION COMPLETE")
print("=" * 70)

print(f"Environment     : {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Content version : {content_version}")
print(f"Format version  : {format_version}")
print(f"Git commit      : {git_commit}")
print(f"Pin             : {PIN if PIN else '(none — latest)'}")
print(f"Snapshot folder : {VERSION_DIR.name}")
print(f"Counts          : {counts}")

print("\nFile sha256:")
for name, info in manifest["files"].items():
    print(f"  {name:18s} {info['sha256']}  {info['path']}")

print("\nSource YAML:")
print(f"  {source_yaml_versioned}")

print("\nATLAS STIX (versioned / latest):")
print(f"  {atlas_stix_versioned}")
print(f"  {atlas_stix_latest}")

print("\nATLAS + ATT&CK Enterprise STIX (versioned / latest):")
print(f"  {atlas_attack_stix_versioned}")
print(f"  {atlas_attack_stix_latest}")

In [ ]:
# Cell 4 — Extract and print the inferred JSON structure

import json

JSON_FILES = {
    "ATLAS_STIX": OUTPUT_DIR / "ATLAS-STIX-latest.json",
    "ATLAS_ATTACK_Enterprise_STIX": (
        OUTPUT_DIR / "ATLAS-ATTACK-Enterprise-STIX-latest.json"
    ),
}


def infer_structure(obj):
    """
    Extract the recursive structure of a JSON document.

    Rules:
    - Field names are preserved.
    - Values are represented by their data types.
    - Nested objects are recursively expanded.
    - Arrays show the structure/type of their elements.
    - Actual values are not included.
    """

    if isinstance(obj, dict):
        return {
            key: infer_structure(value)
            for key, value in obj.items()
        }

    if isinstance(obj, list):
        if not obj:
            return ["unknown"]

        if all(isinstance(item, dict) for item in obj):
            merged = {}

            for item in obj:
                item_structure = infer_structure(item)

                for key, structure in item_structure.items():
                    if key not in merged:
                        merged[key] = structure

            return [merged]

        types = sorted({
            type(item).__name__
            for item in obj
        })

        return [
            types[0]
            if len(types) == 1
            else types
        ]

    if obj is None:
        return "null"

    if isinstance(obj, bool):
        return "boolean"

    if isinstance(obj, int):
        return "integer"

    if isinstance(obj, float):
        return "number"

    if isinstance(obj, str):
        return "string"

    return type(obj).__name__


# ------------------------------------------------------------
# Analyze generated STIX files
# ------------------------------------------------------------

structures = {}

for name, json_file in JSON_FILES.items():

    if not json_file.exists():
        raise FileNotFoundError(
            f"Required JSON file was not found:\n{json_file}"
        )

    print(f"Analyzing: {json_file}")

    with json_file.open("r", encoding="utf-8") as f:
        data = json.load(f)

    structures[name] = infer_structure(data)


# ------------------------------------------------------------
# Print inferred structures
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INFERRED JSON STRUCTURE")
print("=" * 70)

print(
    json.dumps(
        structures,
        indent=2,
        ensure_ascii=False,
    )
)